## @tool装饰器

In [14]:
from langchain_core.tools import tool, StructuredTool
from numba.core.cgutils import Structure
from pydantic import BaseModel, Field


class FieldInfo(BaseModel):
    a: int = Field(description='参数1')
    b: int = Field(description='参数2')


'''
    函数描述信息必须填写
    LLM 会利用函数名和 docstring 来决定何时调用
'''


@tool(name_or_callable='add_two_sum', description='add two',
      return_direct=True, args_schema=FieldInfo)
def add_number(a: int, b: int) -> int:
    """计算两个整数的和"""
    return a + b


print(f"name={add_number.name}")  #默认是函数的名称
print(f"args={add_number.args}")  #参数描述
print(f"description={add_number.description}")  #默认是函数的说明信息
print(f"return_direct={add_number.return_direct}")  #默认值是False

name=add_two_sum
args={'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
description=add two
return_direct=True


`举例二`：进阶

In [14]:
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langchain.agents import create_agent
from langchain_core.tools import tool
import os
import dotenv

'''
create_react_agent已经作废，换成create_agent
'''
dotenv.load_dotenv(dotenv_path='../chapter02-model IO/.env')
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY2")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL2")
llm = ChatOpenAI(
    model='qwen3-vl-flash-2026-01-22',

    temperature=0.6
)


@tool
def calculate_multiply(a: int, b: int) -> int:
    """计算两个数字的乘积。当需要数学运算时使用此工具。"""
    return a * b


# 定义工具列表
tools = [calculate_multiply]

# 创建一个自动执行循环的 Agent
# 它会自动解析 -> 执行工具 -> 获取结果 -> 继续下一步
agent_executor = create_agent(model=llm, tools=tools)

# 执行
result = agent_executor.invoke({"messages": [("human", "42 * 10+1 是多少？")]})

print(result["messages"][-1].content)
# 输出: 42 * 10 是 420。

42 * 10 + 1 = 420 + 1 = 421。


## StructuredTool的from_function()

In [11]:
'''
StructuredTool 是 LangChain 中定义复杂工具的底层类，它允许你通过定义参数的 JSON Schema，强制 LLM 严格遵守输入参数的格式。
虽然 @tool 装饰器是创建工具的最常用方法（它底层就是调用的 StructuredTool），但在某些需要动态生成工具、或需要通过类方法精细控制的场景下，
直接使用 StructuredTool 是更健壮的选择。
'''

# TODO 备注
'''
    什么时候用它？
    当你需要参数约束：比如参数必须在一定范围内，或者是特定的枚举值。
    当函数不是你写的：你不能通过装饰器修改外部函数的代码。
    需要动态工具：需要在运行时动态创建工具逻辑。
'''
# 举例1
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field


class FieldInfo(BaseModel):
    query:str=Field(description='要搜索的关键字')

# 1. 定义一个简单的处理函数
def multiply(query:str) -> str:
    """查询baidu的结果"""
    return '查询的结果'


# 2. 使用 StructuredTool 手动定义 Schema
# 这比直接写 @tool 有更强的灵活性，比如手动指定参数描述
multiply_tool = StructuredTool.from_function(
    func=multiply,
    name="multiply_tool",
    description="用于执行查询网页的工具",
    # True当工具调用之后直接把结果返回给用户，False则把决定权交给Agent，默认False
    # return_direct=True
    # 你甚至可以手动定义 args_schema 来精细控制参数校验
    args_schema=FieldInfo
)
print(f'name={multiply_tool.name}')
print(f'args={multiply_tool.args}')
print(f'description={multiply_tool.description}')
print(f'return_direct={multiply_tool.return_direct}')
# 使用
print(multiply_tool.invoke({'query':'明天天气如何'}))

name=multiply_tool
args={'query': {'description': '要搜索的关键字', 'title': 'Query', 'type': 'string'}}
description=用于执行查询网页的工具
return_direct=False
查询的结果


### 调用工具过程使用与分析

In [2]:
# 大模型会自动分析用户需求，判断是否需要调用指定工具。
# 如果模型认为需要调用工具（如 MoveFileTool ），返回的 message 会包含
# content : 通常为空（因为模型选择调用工具，而非生成自然语言回复）。
# additional_kwargs : 包含工具调用的详细信息：
# 如果模型认为无需调用工具（例如用户输入与工具无关），返回的 message 会是普通文本回复

from datetime import date
from langchain_core.tools import tool
from llm.my_llm import model
from loguru import logger
# 如果 loguru 重复输出，可以先清理 handler（仅在该单元格运行一次即可）
logger.remove()
logger.add(lambda msg: print(msg, end=''), level="INFO")
@tool
def get_today()->str:
    """
    获取当前系统日期

    Returns:
        str: 今天的日期字符串，格式为 yyyy-MM-dd
    """
    logger.info('执行工具：get_today')
    return date.today().isoformat()

tools_map={'get_today':get_today}
# 将工具绑定到语音模型
llm_with_tools=model.bind_tools(list(tools_map.values()))

# 用户提问
question_list=['你是谁','今天谁几号？']

for question in question_list:
    logger.info(f'用户问题：{question}')
    # 调用语言模型处理用户问题
    ai_msg=llm_with_tools.invoke(question)
    logger.info(f'LLM回复：{ai_msg}')
    # 检查是否有工具调用
    if ai_msg.tool_calls:
        # logger.info(ai_msg.tool_calls)
        # 获取第一个工具调用信息
        tool_call=ai_msg.tool_calls[0]
        # 执行对应的工具函数并获取结果
        tool_result=locals()[tool_call['name']].invoke(tool_call['args'])
        logger.info(f'调用工具结果：{tool_result}')
    else:
        # 直接输出语言模型的回答
        logger.info(f'LLM直接作答：{ai_msg.content}')









2026-06-10 18:47:40.878 | INFO     | __main__:<module>:33 - 用户问题：你是谁
2026-06-10 18:47:46.017 | INFO     | __main__:<module>:36 - LLM回复：content='我是通义千问（Qwen），是由阿里巴巴集团通义实验室自主研发的大型语言模型。很高兴为您服务！请问有什么我可以帮您的吗？' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 252, 'prompt_tokens': 270, 'total_tokens': 522, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 217, 'rejected_prediction_tokens': None, 'text_tokens': 252}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 270}}, 'model_provider': 'openai', 'model_name': 'qwen3.7-plus', 'system_fingerprint': None, 'id': 'chatcmpl-4b270683-fd23-92fb-83be-644170d4b865', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019eb125-204f-7c73-ac34-7b026c5c3a18-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 270, 'output_tokens': 252, 'total_tokens': 522, 'input_token_details': {'cache_read': 0}, '

### 定义Tool

In [3]:
from langchain_core.tools import tool
from langchain_core.output_parsers import JsonOutputKeyToolsParser,StrOutputParser
from langchain_core.prompts import PromptTemplate
from loguru import logger
from llm.my_llm import model
import json
import os
import httpx

@tool
def get_weather(loc):
    """
    查询即时天气函数

    :param loc: 必要参数，字符串类型，用于表示查询天气的具体城市名称。
                注意，中国的城市需要用对应城市的英文名称代替，例如如果需要查询北京市天气，
                则 loc 参数需要输入 '北京'。
    :return: OpenWeather API 查询即时天气的结果。具体 URL 请求地址为：
             https://api.openweathermap.org/data/2.5/weather。
             返回结果对象类型为解析之后的 JSON 格式对象，并用字符串形式进行表示，
             其中包含了全部重要的天气信息。
    """
    url=f"https://wttr.in/{loc}?format=j1"
    response=httpx.get(url)
    data=response.json()
    return json.dumps(data)

llm_with_tools=model.bind_tools([get_weather])

# 创建解析器，用于提取工具调用结果中的 JSON 数据
parser=JsonOutputKeyToolsParser(key_name=get_weather.name,first_tool_only=True)

# 构建工具调用链：模型 -> 解析器 -> 调用天气工具
get_weather_chain=llm_with_tools|parser|get_weather

output_prompt=PromptTemplate.from_template(
    """你将收到一段 JSON 格式的天气数据{weather_json}，请用简洁自然的方式将其转述给用户。
    以下是天气 JSON 数据：
    请将其转换为中文天气描述，例如：
    “北京现在天气：多云，气温 28℃，体感有点闷热（约 32℃），湿度 75%，微风（东南风 2 米/秒），能见度很好，大约 10 公里。建议穿短袖短裤。适合做户外运动。"
    """
)
# 创建字符串输出解析器
output_parser=StrOutputParser()

output_chain=output_prompt|model|output_parser

# 构建完整的处理链：天气查询链 ->将天气数据包装为字典格式 -> 输出链
full_chain=get_weather_chain | (lambda x:{"weather_json":x})|output_chain


# 执行完整链路，查询上海天气并打印结果
result=full_chain.invoke('请问上海今天的天气如何')
logger.info(result)



2026-06-10 19:00:52.089 | INFO     | __main__:<module>:54 - 上海浦东现在天气：晴，气温 29℃，体感舒适（约 29℃），湿度 48%，东南风（约 4.4 米/秒），能见度很好，大约 10 公里。今天最高气温 29℃，最低气温 19℃。建议穿短袖短裤等夏装。白天阳光充足，非常适合户外活动，但中午紫外线较强，外出请注意防晒。
